# LAB S08: Proyecto final, implementación guiada

El cuaderno solo importa y ejecuta lo que el equipo ya unió en su repositorio (carpeta `pf`) y usa `mi_pipeline` del clon personal. Cada control (C0 a C5) confirma una etapa del trabajo: entorno, ciclo corto, pieza real con pruebas, conflicto resuelto, salidas JSON del flujo y exactitud por fase. Los documentos son ficticios y cada uno trae su respuesta correcta para medir. Ningún documento sale de la VM. Esta sesión no usa modelos de lenguaje.

## 0. Dónde están los datos de práctica

In [ ]:
# Celda estándar de localización de datos (no se modifica)
import os
from pathlib import Path


def localizar_carpeta(nombre="02_Datos_Practica", max_niveles=6, obligatorio=True):
    """Busca `nombre` en la carpeta actual y en sus carpetas superiores."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents][: max_niveles + 1]:
        candidata = carpeta / nombre
        if candidata.is_dir():
            return candidata
    # Respaldo: carpetas del curso en la unidad del sistema (VM de laboratorio)
    curso = Path(os.environ.get("SystemDrive", "C:") + os.sep) / "curso"
    respaldo = [curso / "paquete" / nombre]
    if nombre == "02_Datos_Practica":
        respaldo.insert(0, curso / "datos")
    for candidata in respaldo:
        if candidata.is_dir():
            return candidata
    if obligatorio:
        raise FileNotFoundError(
            f"No se encuentra la carpeta '{nombre}'. El cuaderno se abre desde su carpeta "
            "dentro del repositorio del curso, sin moverlo de lugar."
        )
    return None


DATOS = localizar_carpeta("02_Datos_Practica")
print("Datos de práctica encontrados:", DATOS.name)
print("Subcarpetas:", sorted(p.name for p in DATOS.iterdir() if p.is_dir()))


## Paso 0: conexión con `mi_pipeline` y con el repositorio del equipo (C0)

In [ ]:
# Conexión de mi_pipeline y de las herramientas del LAB (no se modifica)
%load_ext autoreload
%autoreload 2
import html
import json
import re
import shutil
import subprocess
import sys
import tempfile
import time
import unicodedata
import warnings

import pandas as pd
import pymupdf
import pytesseract
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)   # avisos de cambios futuros de las bibliotecas, que no cambian el resultado
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", None)


def localizar_raiz(max_niveles=6):
    """Carpeta del repositorio clonado que contiene mi_pipeline."""
    if os.environ.get("MI_PIPELINE_RAIZ"):
        return Path(os.environ["MI_PIPELINE_RAIZ"])
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents][: max_niveles + 1]:
        if (carpeta / "mi_pipeline" / "__init__.py").is_file():
            return carpeta
    clon = Path(os.environ.get("SystemDrive", "C:") + os.sep) / "curso" / "trabajo" / "entorno-extraccion-documental"
    if (clon / "mi_pipeline" / "__init__.py").is_file():
        return clon
    raise FileNotFoundError("No se encuentra la carpeta mi_pipeline: la preparación de la sesión la repone.")


RAIZ = localizar_raiz()
sys.path.insert(0, str(RAIZ))
SALIDAS = Path("salidas").resolve()
SALIDAS.mkdir(exist_ok=True)
CONTROLES = []
INICIO = time.perf_counter()
pytesseract.pytesseract.tesseract_cmd = (
    os.environ.get("TESSERACT_CMD")
    or shutil.which("tesseract")
    or str(Path(os.environ.get("ProgramFiles", "")) / "Tesseract-OCR" / "tesseract.exe")
)


def correr_pruebas():
    """Corre las pruebas de mi_pipeline con pytest, muestra el resumen y dice si todas pasaron."""
    corrida = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider",
         "--basetemp", tempfile.mkdtemp(prefix="pytest_s08_"), str(RAIZ / "mi_pipeline" / "tests")],
        capture_output=True, text=True, cwd=str(RAIZ), timeout=900, encoding="utf-8", errors="replace",
    )
    resumen = [linea for linea in corrida.stdout.strip().splitlines() if " passed" in linea or " failed" in linea or "error" in linea]
    print(re.sub(r" in [\d.]+s.*", "", resumen[-1]) if resumen else corrida.stdout[-500:])
    return corrida.returncode == 0


# Cómo se muestran los resultados: tablas alineadas, con el texto completo y los números como se escriben en el Perú
PORCENTAJES = {"exactitud"}
SIN_MILES = {"contexto", "nivel"}


def num(valor, decimales=2):
    """Un número con punto de miles, coma decimal y sin ceros de más: 7.728 o 10,78."""
    texto = f"{valor:,.{decimales}f}".replace(",", "§").replace(".", ",").replace("§", ".")
    return texto.rstrip("0").rstrip(",") if "," in texto else texto


def pct(valor):
    """Una proporción como porcentaje: 95,8 %."""
    return f"{valor * 100:.1f} %".replace(".", ",")


def si_no(valor):
    return "sí" if valor else "no"


def como_texto(valor, columna="", vacio="vacío"):
    """Un valor tal como se muestra en una tabla: sí o no, números en formato local y «vacío» si no hay dato."""
    if pd.api.types.is_scalar(valor) and pd.isna(valor):
        return vacio
    if hasattr(valor, "item"):
        valor = valor.item()
    if isinstance(valor, bool):
        return si_no(valor)
    if isinstance(valor, (int, float)):
        if columna in PORCENTAJES:
            return pct(valor)
        if columna in SIN_MILES:
            return str(valor)
        return num(valor, 3 if "CER" in columna else 2)
    return str(valor)


def ver(tabla, titulo, vacio="vacío"):
    """Muestra una tabla alineada y con el texto completo; la tabla original no cambia."""
    if tabla.empty:
        print(f"{titulo}. Ninguno.")
        return
    base = tabla.reset_index() if tabla.index.name else tabla
    vista = pd.DataFrame({columna: [como_texto(v, str(columna), vacio) for v in base[columna]] for columna in base.columns})
    cuerpo = (vista.to_html(index=False, border=0)
              .replace("<th>", '<th style="text-align:left;vertical-align:bottom;padding:4px 10px">')
              .replace("<td>", '<td style="text-align:left;vertical-align:top;white-space:normal;padding:4px 10px">'))
    display({"text/html": f"<p><b>{html.escape(titulo)}</b></p>{cuerpo}", "text/plain": titulo + "\n" + vista.to_string(index=False)}, raw=True)


def decir(texto):
    """Dice qué mostró esta corrida, con sus propios números."""
    display({"text/html": f'<p style="border-left:4px solid #888;padding:6px 10px"><b>En esta corrida:</b> {html.escape(texto)}</p>',
             "text/plain": "En esta corrida: " + texto}, raw=True)


# Medición contra la respuesta correcta de cada documento (la misma vara en todos los ejercicios)
NO_SE_MIDEN = {"ambiguedad", "cuadra", "paginas", "n_paginas", "monto_en_texto", "canal", "adjunto",
               "n_requisitos", "n_requisitos_cumplen", "n_requisitos_no_cumplen", "n_requisitos_observados"}
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "setiembre", "octubre", "noviembre", "diciembre"]


def llano(texto):
    """Minúsculas, sin tildes y con un solo espacio."""
    base = "".join(c for c in unicodedata.normalize("NFD", str(texto)) if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", " ", base.lower()).strip(" .,:;")


def figura(valor, texto):
    """Dice si una fecha (en cualquiera de sus formas escritas) o un código figura en el texto."""
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", valor):
        anio, mes, dia = valor.split("-")
        meses = [MESES[int(mes) - 1]] + (["septiembre"] if mes == "09" else [])
        formas = [valor, f"{dia}/{mes}/{anio}", f"{int(dia)}/{mes}/{anio}"] + [f"{d} de {m} de {anio}" for m in meses for d in (dia, int(dia))]
        return any(forma in texto for forma in formas)
    return llano(valor) in texto


def correcto_de(id_):
    """Respuesta correcta de un documento: sus campos, con null en las fechas y códigos que no figuran en su texto."""
    datos = json.loads((DATOS / "10_Ground_Truth" / f"{id_}.json").read_text(encoding="utf-8"))
    texto = llano(datos["texto_completo"])
    correcto = {}
    for campo, valor in datos["campos"].items():
        if campo in NO_SE_MIDEN or isinstance(valor, (bool, dict, list)):
            continue
        if isinstance(valor, str) and (re.fullmatch(r"\d{4}-\d{2}-\d{2}", valor) or campo == "numero_reclamo") and not figura(valor, texto):
            valor = None
        correcto[campo] = valor
    return correcto


def es_correcto(valor, correcto):
    """Medida estricta: el dato listo para usar (número como número, fecha AAAA-MM-DD, moneda PEN); null es correcto si no hay dato."""
    if correcto is None:
        return valor is None
    if valor is None:
        return False
    if isinstance(correcto, (int, float)):
        return isinstance(valor, (int, float)) and not isinstance(valor, bool) and abs(valor - correcto) < 0.011
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}|\d{8,20}", str(correcto)):
        return str(valor).strip() == correcto
    return llano(valor) == llano(correcto)


def aciertos(extraido, correcto):
    """(correctos, esperados, fallos) de un documento."""
    escrito = lambda v: "vacío" if v is None else repr(v)
    fallos = [f"{campo}: {escrito(extraido.get(campo))} en lugar de {escrito(valor)}" for campo, valor in correcto.items() if not es_correcto(extraido.get(campo), valor)]
    return len(correcto) - len(fallos), len(correcto), fallos


# S08: el repositorio del equipo (carpeta pf) y mi_pipeline del clon personal
os.environ["TESSERACT_CMD"] = pytesseract.pytesseract.tesseract_cmd
os.environ["PYTHONPATH"] = os.pathsep.join([str(RAIZ), os.environ.get("PYTHONPATH", "")])   # la Terminal y las pruebas encuentran mi_pipeline


def localizar_equipo(max_niveles=6):
    """Raíz del repositorio del equipo: la carpeta que contiene pf; sube desde la carpeta del cuaderno o mira la carpeta curso/trabajo del disco del sistema."""
    if os.environ.get("EQUIPO_RAIZ"):
        return Path(os.environ["EQUIPO_RAIZ"])
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents][: max_niveles + 1]:
        if (carpeta / "pf" / "__init__.py").is_file():
            return carpeta
    trabajo = Path(os.environ.get("SystemDrive", "C:") + os.sep) / "curso" / "trabajo"
    for carpeta in sorted(trabajo.glob("*")) if trabajo.is_dir() else []:
        if (carpeta / "pf" / "__init__.py").is_file():
            return carpeta
    return None


EQUIPO = localizar_equipo()
if EQUIPO is not None:
    os.chdir(EQUIPO)             # desde aquí las rutas del equipo son relativas: pf/integracion/pipeline_equipo.yaml
    sys.path.insert(0, str(EQUIPO))


def correr_pruebas_equipo(carpeta="pf"):
    """Corre las pruebas de una carpeta del repositorio del equipo y dice si todas pasaron."""
    corrida = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", carpeta],
                             capture_output=True, text=True, cwd=str(EQUIPO), timeout=900, encoding="utf-8", errors="replace")
    resumen = [linea for linea in corrida.stdout.strip().splitlines() if " passed" in linea or " failed" in linea or "error" in linea]
    print(carpeta + ":", re.sub(r" in [\d.]+s.*", "", resumen[-1]) if resumen else corrida.stdout[-500:])
    return corrida.returncode == 0


print("Carpeta del repositorio:", RAIZ.name, "| Tesseract:", pytesseract.get_tesseract_version())

In [ ]:
# ✔ Control C0: el entorno, el clon personal y el repositorio del equipo
import mi_pipeline   # el clon personal (S03 a S06) debe estar en el camino de Python
git_version = subprocess.run(["git", "--version"], capture_output=True, text=True).stdout.strip()
print(git_version, "| mi_pipeline:", Path(mi_pipeline.__file__).resolve().parent.name, "| clon personal:", RAIZ.name)
if EQUIPO is None:
    print("AVISO: no se encuentra el repositorio del equipo (la carpeta que contiene pf). Se clona o se abre el cuaderno desde su carpeta; ver la guía.")
else:
    rama = subprocess.run(["git", "branch", "--show-current"], cwd=EQUIPO, capture_output=True, text=True).stdout.strip()
    print("Repositorio del equipo:", EQUIPO.name, "| rama:", rama)
assert git_version.startswith("git version"), "C0: falta Git."
assert pytesseract.get_tesseract_version() is not None, "C0: falta Tesseract."
CONTROLES.append("C0")
print("✔ C0 superado: Git, Tesseract y mi_pipeline listos")

## E1. El ciclo corto: una persona, una carpeta, una rama, un pull request (C1)

◆ **El mapa.** Cada función trabaja en su carpeta `pf/<función>` y en su rama; el pull request presenta el expediente para la revisión de un compañero.

In [ ]:
# E1: el ciclo corto subió y se unió (C1). Cada función tiene su carpeta y su LEEME.txt en el repositorio del equipo
assert EQUIPO is not None, "C1: falta el repositorio del equipo; se actualiza con git pull de la rama pf-s08."
FUNCIONES = {"integracion": "Líder de integración", "extraccion": "Responsable de extracción", "calidad": "Responsable de calidad", "documentacion": "Documentación y presentación"}
filas = []
for carpeta, funcion in FUNCIONES.items():
    leeme = EQUIPO / "pf" / carpeta / "LEEME.txt"
    filas.append({"función": funcion, "carpeta": f"pf/{carpeta}", "LEEME.txt": leeme.is_file(), "primera línea": leeme.read_text(encoding="utf-8").splitlines()[0] if leeme.is_file() else ""})
ver(pd.DataFrame(filas), "Lo que el equipo ya unió en pf-s08")
historia = subprocess.run(["git", "log", "--oneline", "-n", "10"], cwd=EQUIPO, capture_output=True, text=True).stdout
print("Últimos avances guardados:\n" + historia)
assert all(fila["LEEME.txt"] for fila in filas), "C1: falta el LEEME.txt de alguna función; su pull request aún no se unió."
assert (EQUIPO / "pf" / "__init__.py").is_file() and (EQUIPO / ".gitignore").is_file(), "C1: faltan pf/__init__.py o .gitignore (los crea el líder)."
CONTROLES.append("C1")
print("✔ C1 superado: las 4 carpetas del equipo están unidas")

## E2. La pieza real de cada función (C2)

◆ **Revisar un pull request.** Cuatro preguntas: ¿sube solo lo de su carpeta?, ¿pasan sus pruebas?, ¿hay claves o archivos pesados?, ¿se entiende su LEEME?

In [ ]:
# E2: la pieza real de cada función, con sus pruebas, y una muestra de lo que hacen juntas (C2)
import yaml
from pf.calidad.validadores import ruc_valido, validar_campos
from pf.extraccion.extractor_flujo import extraer_documento

CFG_EQUIPO = yaml.safe_load((EQUIPO / "pf" / "integracion" / "pipeline_equipo.yaml").read_text(encoding="utf-8"))
FLUJO = CFG_EQUIPO["flujo"]
CARPETAS = {"A": "06_Reclamos", "B": "02_Oficios", "C": "03_Estados_Financieros", "D": "04_Contratos"}
muestra = sorted(p for p in (DATOS / CARPETAS[FLUJO]).iterdir() if p.is_file())[0]
campos = extraer_documento(muestra, FLUJO)
ver(pd.DataFrame([{"campo": campo, "valor": valor} for campo, valor in campos.items()]), f"Campos que el extractor entrega para {muestra.name} (flujo {FLUJO})")
problemas = validar_campos(campos, FLUJO)
print("Validación:", problemas if problemas else "sin problemas")
print("RUC de la muestra válido:", ruc_valido(campos.get("ruc")))
ok_extraccion = correr_pruebas_equipo("pf/extraccion")
ok_calidad = correr_pruebas_equipo("pf/calidad")
assert ok_extraccion and ok_calidad, "C2: alguna prueba de pf falla; se corrige en la rama de la función y se vuelve a subir."
CONTROLES.append("C2")
print("✔ C2 superado: la pieza de extracción y la de calidad pasan sus pruebas")

## E3. Integración con el conflicto planeado (C3)

In [ ]:
# E3: el conflicto de la línea etapas quedó resuelto y la configuración del equipo es válida (C3)
from pf.integracion.integrar import leer_config

texto_yaml = (EQUIPO / "pf" / "integracion" / "pipeline_equipo.yaml").read_text(encoding="utf-8")
print([linea for linea in texto_yaml.splitlines() if linea.startswith("etapas:")])
assert not any(marca in texto_yaml for marca in ("<<<<<<<", "=======", ">>>>>>>")), "C3: quedan marcas de conflicto en pipeline_equipo.yaml."
CFG = leer_config(EQUIPO / "pf" / "integracion" / "pipeline_equipo.yaml")
CFG["carpeta_lote"] = str(DATOS)
ver(pd.DataFrame([{"clave": clave, "valor": str(CFG[clave])} for clave in ("flujo", "carpeta_lote", "n_documentos", "salida", "etapas")]), "Lo que dice el YAML del equipo")
assert CFG["etapas"] == ["ingesta", "ocr", "pln", "extraccion", "validacion"], "C3: la línea etapas debe quedar con las cinco etapas, en este orden."
CONTROLES.append("C3")
print("✔ C3 superado: conflicto resuelto y YAML válido")

## Caso: la cadena completa del equipo y sus salidas JSON (C4)

In [ ]:
# Caso: la cadena del equipo sobre el lote del flujo y las salidas JSON con el contrato del proyecto final (C4)
from pf.integracion.integrar import procesar_flujo

inicio_lote = time.perf_counter()
resultados = procesar_flujo(CFG)
segundos_lote = time.perf_counter() - inicio_lote
tabla_lote = pd.DataFrame([{"documento": r["doc_id"], "campos llenos": sum(v is not None for v in r["campos"].values()), "campos": len(r["campos"]),
                            "campos vacíos": ", ".join(r["_vacios"]), "validaciones que fallan": len(r["_problemas"]),
                            "revisión humana": r["revision_humana"], "segundos": r["tiempo_s"]} for r in resultados])
ver(tabla_lote, f"El lote del flujo {CFG['flujo']}")
archivos_json = sorted((EQUIPO / CFG["salida"]).glob("*.json"))
CLAVES = {"doc_id", "campos", "revision_humana", "tiempo_s", "estrategia"}
contenidos = [json.loads(a.read_text(encoding="utf-8")) for a in archivos_json]
print(json.dumps(contenidos[0], ensure_ascii=False, indent=1)[:900])
con_revision = sum(c["revision_humana"] for c in contenidos)
decir(f"el lote de {len(resultados)} documentos tardó {num(segundos_lote, 1)} s; {con_revision} piden revisión humana ({pct(con_revision / len(contenidos))}).")
assert len(archivos_json) == CFG["n_documentos"], f"C4: se esperaban {CFG['n_documentos']} salidas JSON y hay {len(archivos_json)}."
assert all(set(c) == CLAVES and isinstance(c["revision_humana"], bool) for c in contenidos), "C4: alguna salida no cumple el contrato (doc_id, campos, revision_humana, tiempo_s, estrategia)."
assert all(c["revision_humana"] for c in contenidos if any(v is None for k, v in c["campos"].items() if k not in CFG["campos_opcionales"])), "C4: un documento con campos vacíos debe ir a revisión humana."
CONTROLES.append("C4")
print("✔ C4 superado:", len(archivos_json), "salidas JSON con el contrato del proyecto final")

## Cierre: exactitud por fase y por campo (C5)

In [ ]:
# Cierre: la exactitud después de cada fase y por campo (C5)
from pf.calidad.medir_exactitud import escribir_csv, exactitud_por_campo, leer_salidas

GT = DATOS / "10_Ground_Truth"
FASES = {"ingesta": ["ingesta", "extraccion"], "ingesta + OCR": ["ingesta", "ocr", "extraccion"], "ingesta + OCR + PLN": ["ingesta", "ocr", "pln", "extraccion", "validacion"]}
filas_fase = []
for fase, etapas in FASES.items():
    cfg_fase = {**CFG, "etapas": etapas, "salida": str(SALIDAS / ("fase_" + str(len(etapas)) + "_" + fase.replace(" ", "_").replace("+", "mas")))}
    if etapas == FASES["ingesta + OCR + PLN"]:
        cfg_fase["salida"] = CFG["salida"]              # la corrida completa ya se hizo en C4: se mide su salida
    else:
        procesar_flujo(cfg_fase)
    tabla = exactitud_por_campo(leer_salidas(cfg_fase["salida"]), GT)
    total = [fila for fila in tabla if fila["campo"] == "todos"]
    filas_fase.append({"fase": fase, "correctos": sum(f["correctos"] for f in total), "esperados": sum(f["esperados"] for f in total)})
    filas_fase[-1]["exactitud"] = filas_fase[-1]["correctos"] / filas_fase[-1]["esperados"]
ver(pd.DataFrame(filas_fase), "Exactitud después de cada fase")
tabla_campos = [fila for fila in tabla if fila["campo"] != "todos"]
ver(pd.DataFrame(tabla_campos).sort_values("exactitud"), "Exactitud por campo con la cadena completa (de menor a mayor)")
escribir_csv(tabla, EQUIPO / "pf" / "calidad" / "exactitud_por_campo.csv")
UMBRAL = {"A": 0.85, "B": 0.80, "C": 0.85, "D": 0.80}[CFG["flujo"]]
exactitudes = [fila["exactitud"] for fila in filas_fase]
decir(f"la exactitud pasó de {pct(exactitudes[0])} con solo la ingesta a {pct(exactitudes[-1])} con la cadena completa; el campo más débil es "
      f"{min(tabla_campos, key=lambda f: f['exactitud'])['campo']}.")
assert exactitudes == sorted(exactitudes), "C5: la exactitud no debe bajar al agregar una fase."
assert exactitudes[-1] >= UMBRAL, f"C5: la exactitud final {pct(exactitudes[-1])} queda bajo el umbral {pct(UMBRAL)} del flujo."
CONTROLES.append("C5")
print("✔ C5 superado: exactitud por fase y por campo; archivo pf/calidad/exactitud_por_campo.csv escrito")

In [ ]:
# Resumen de los controles y del tiempo del cuaderno
ver(pd.DataFrame({"control": CONTROLES}), "Controles superados")
assert CONTROLES == ["C0", "C1", "C2", "C3", "C4", "C5"], "Faltan controles por superar."
print(f"El cuaderno tardó {num((time.perf_counter() - INICIO) / 60, 1)} minutos.")

## Reto extra

El participante que termina antes cambia `n_documentos` a 10 en una copia de la configuración y compara los segundos del lote con los de 20 documentos; después cambia `flujo` a otro (A, B, C o D) en esa copia y observa qué campo queda más débil.